In [5]:
from langchain_community.retrievers import WikipediaRetriever

In [16]:
retriever = WikipediaRetriever(top_k_results=10, lang="en")

In [17]:
# Define your query
query = "the geopolitical history of india and pakistan from the perspective of a chinese"

In [19]:
docs = retriever.invoke(query)

In [21]:
docs

[Document(metadata={'title': 'India–Pakistan war of 1971', 'summary': "The India–Pakistan war of 1971, also known as the third Indo-Pakistani war, was a military confrontation between India and Pakistan that occurred during the Bangladesh Liberation War in East Pakistan from 3 December 1971 until the Pakistani capitulation in Dhaka on 16 December 1971.  The war began with Pakistan's Operation Chengiz Khan, consisting of preemptive aerial strikes on eight Indian air stations. The strikes led to India declaring war on Pakistan, marking their entry into the war for East Pakistan's independence, on the side of Bengali nationalist forces. India's entry expanded the existing conflict with Indian and Pakistani forces engaging on both the eastern and western fronts.\nThirteen days after the war started, India achieved a clear advantage, and the Eastern Command of the Pakistan military signed the instrument of surrender on 16 December 1971 in Dhaka, marking the formation of East Pakistan as the

In [22]:
# Print retrieved content
for i, doc in enumerate(docs):
    print(f"\n--- Result {i+1} ---")
    print(f"Content:\n{doc.page_content}...")  # truncate for display


--- Result 1 ---
Content:
The India–Pakistan war of 1971, also known as the third Indo-Pakistani war, was a military confrontation between India and Pakistan that occurred during the Bangladesh Liberation War in East Pakistan from 3 December 1971 until the Pakistani capitulation in Dhaka on 16 December 1971.  The war began with Pakistan's Operation Chengiz Khan, consisting of preemptive aerial strikes on eight Indian air stations. The strikes led to India declaring war on Pakistan, marking their entry into the war for East Pakistan's independence, on the side of Bengali nationalist forces. India's entry expanded the existing conflict with Indian and Pakistani forces engaging on both the eastern and western fronts.
Thirteen days after the war started, India achieved a clear advantage, and the Eastern Command of the Pakistan military signed the instrument of surrender on 16 December 1971 in Dhaka, marking the formation of East Pakistan as the new nation of Bangladesh. Approximately 93,0

# Vectore Store Retriever 

In [23]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

from dotenv import load_dotenv

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/mnt/c/Users/vinay kumar/Desktop/LangChain_and_RAG/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2555.20it/s]


In [45]:
# Step 1: Your source documents
from langchain_core.documents import Document 
documents = [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="Chroma is a vector database optimized for LLM-based search."),
    Document(page_content="Embeddings convert text into high-dimensional vectors and chroma ."),
    Document(page_content="OpenAI provides powerful embedding models."),
]

In [46]:
# Creating a chroma vectore Store 
vectorestore = Chroma.from_documents(
  documents=documents,
  embedding=embeddings,
  collection_name="my_collection"
)

In [48]:
retriver = vectorestore.as_retriever(search_kwargs={"k":2})

In [49]:
query = "What is Chroma used for?"
result = retriever.invoke(query)

In [51]:
for i , doc in enumerate(result):
  print(f"\n Result {i+1} -----")
  print(doc.page_content)


 Result 1 -----
Chroma key compositing, or chroma keying, is a visual-effects and post-production technique for compositing (layering) two or more images or video streams together based on colour hues (chroma range). The technique has been used in many fields to remove a background from the subject of a photo or video — particularly the newscasting, motion picture, and video game industries. A colour range in the foreground footage is made transparent, allowing separately filmed background footage or a static image to be inserted into the scene. The chroma keying technique is commonly used in video production and post-production. This technique is also referred to as colour keying or colour separation overlay (CSO; primarily by the BBC), or by various terms for specific colour-related variants such as green screen or blue screen; chroma keying can be done with backgrounds of any colour that are uniform and distinct, but green and blue backgrounds are more commonly used because they di

In [52]:
results = vectorestore.similarity_search(query,k=3)

In [54]:
print(results)

[Document(id='043f91cb-8f5f-49e0-b415-2d91bd159b33', metadata={}, page_content='Chroma is a vector database optimized for LLM-based search.'), Document(id='2dfc0832-ca41-45cd-895b-45df350ab737', metadata={}, page_content='Chroma is a vector database optimized for LLM-based search.'), Document(id='67f2e471-cc02-4497-985b-ad51bcbdca51', metadata={}, page_content='Embeddings convert text into high-dimensional vectors and chroma .')]


# MMR

In [64]:
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

In [65]:
from langchain_community.vectorstores import FAISS

vectorestore = FAISS.from_documents(
  documents=docs,
  embedding=embeddings
)

In [66]:
# Enable MMR in the retriver
retriever = vectorestore.as_retriever(
  search_type = "mmr",
  search_kwargs={"k":3,"lambda_mult":0.5} #lambda_mult = relevance-diversity balance
)

In [68]:
query = "What is the langchain"
result1 = retriever.invoke(query)

In [69]:
for i, doc1 in enumerate(result):
  print(f"\n Result {i+1}----")
  print(doc1.page_content)


 Result 1----
Chroma key compositing, or chroma keying, is a visual-effects and post-production technique for compositing (layering) two or more images or video streams together based on colour hues (chroma range). The technique has been used in many fields to remove a background from the subject of a photo or video — particularly the newscasting, motion picture, and video game industries. A colour range in the foreground footage is made transparent, allowing separately filmed background footage or a static image to be inserted into the scene. The chroma keying technique is commonly used in video production and post-production. This technique is also referred to as colour keying or colour separation overlay (CSO; primarily by the BBC), or by various terms for specific colour-related variants such as green screen or blue screen; chroma keying can be done with backgrounds of any colour that are uniform and distinct, but green and blue backgrounds are more commonly used because they diff

# Multiquery Retriever

In [70]:
from langchain_classic.retrievers import MultiQueryRetriever

In [71]:
# Relevant health & wellness documents
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [83]:
vectorestore = FAISS.from_documents(documents=all_docs,embedding=embeddings)

In [84]:
similarity_retriever = vectorestore.as_retriever(documents=all_docs,embedding=embeddings)

In [85]:
pip install langchain-ollama

Note: you may need to restart the kernel to use updated packages.


In [102]:
from langchain_ollama import ChatOllama
# from bs4 import BeautifulSoup


model = ChatOllama(
    model="llama3.2:latest",
    temperature=0.1
)

In [87]:
multiquery_retriver = MultiQueryRetriever.from_llm(
  retriever=vectorestore.as_retriever(search_kwargs={"k":5}),
  llm=model 
)

In [89]:
query = "How to improve energy levels and maintain balance?"

In [106]:
similarity_results = similarity_retriever.invoke(query)
multiquery_results = multiquery_retriver.invoke(query)

In [107]:
for i, docs in enumerate(similarity_results):
  print(f"\n ----- Result {i+1}---------")
  print(doc.page_content)

  print("*"*100)


  for i, docs in enumerate(multiquery_results):
   print(f"\n ----- Result {i+1}---------")
   print(doc.page_content)

   print("*"*100)


 ----- Result 1---------
A vector database, vector store or vector search engine is a database that stores and retrieves embeddings of data in vector space. Vector databases typically implement approximate nearest neighbor algorithms so users can search for records semantically similar to a given input, unlike traditional databases which primarily look up records by exact match. Use-cases for vector databases include similarity search, semantic search, multi-modal search, recommendations engines, object detection, and retrieval-augmented generation (RAG).
Vector embeddings are mathematical representations of data in a high-dimensional space. In this space, each dimension corresponds to a feature of the data, with the number of dimensions ranging from a few hundred to tens of thousands, depending on the complexity of the data being represented. Each data item is represented by one vector in this space. Words, phrases, or entire documents, as well as images, audio, and other types of da

# ContextualCompressionRetriever

In [108]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_core.documents import Document

In [109]:
# Recreate the document objects from the previous data
docs2 = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
] 

In [110]:
vectorestore2 = FAISS.from_documents(docs2, embeddings)

In [117]:
base_retriever = vectorestore2.as_retriever(search_kwargs={"k":5})

In [118]:
compressor = LLMChainExtractor.from_llm(model)

In [120]:
compression_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor
)

In [121]:
query = "What is photosynthesis?"


In [122]:
compressed_results = compression_retriever.invoke(query)

In [123]:
for i, doc in enumerate(compressed_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Photosynthesis is the process by which green plants convert sunlight into energy.

--- Result 2 ---
The chlorophyll in plant cells captures sunlight during photosynthesis.


In [132]:
for i, doc in enumerate(base_retriever):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years.

--- Result 2 ---
In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls.

--- Result 3 ---
The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design.

--- Result 4 ---
Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league.
